# 导入Python库&模块并配置运行信息

In [1]:
import math
import numpy as np
import pandas as pd
import os
import math
import random
import codecs
from pathlib import Path

import mindspore
import mindspore.dataset as ds
import mindspore.nn as nn
from mindspore import Tensor
from mindspore import context
from mindspore.train.model import Model
from mindspore.nn.metrics import Accuracy
from mindspore.train.serialization import load_checkpoint, load_param_into_net
from mindspore.train.callback import ModelCheckpoint, CheckpointConfig, LossMonitor, TimeMonitor
from mindspore.ops import operations as ops

from easydict import EasyDict as edict


cfg = edict({
    'name': 'movie review',
    'pre_trained': False,
    'num_classes': 2,
    'batch_size': 64,
    'epoch_size': 4,
    'weight_decay': 3e-5,
    'data_path': './TextCNN/data/',
    'device_target': 'CPU',
    'device_id': 0,
    'keep_checkpoint_max': 1,
    'checkpoint_path': './ckpt/train_textcnn-4_149.ckpt',
    'word_len': 51,
    'vec_length': 40
})

context.set_context(mode=context.GRAPH_MODE, device_target=cfg.device_target, device_id=cfg.device_id)

# 数据读取和处理

In [2]:
with open("./TextCNN/data/rt-polarity.neg", 'r', encoding='utf-8') as f:
        print("Negative reivews:")
        for i in range(5):
            print("[{0}]:{1}".format(i,f.readline()))
with open("./TextCNN/data/rt-polarity.pos", 'r', encoding='utf-8') as f:
        print("Positive reivews:")
        for i in range(5):
            print("[{0}]:{1}".format(i,f.readline()))

Negative reivews:
[0]:simplistic , silly and tedious . 

[1]:it's so laddish and juvenile , only teenage boys could possibly find it funny . 

[2]:exploitative and largely devoid of the depth or sophistication that would make watching such a graphic treatment of the crimes bearable . 

[3]:[garbus] discards the potential for pathological study , exhuming instead , the skewed melodrama of the circumstantial situation . 

[4]:a visually flashy but narratively opaque and emotionally vapid exercise in style and mystification . 

Positive reivews:
[0]:the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal . 

[1]:the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson's expanded vision of j . r . r . tolkien's middle-earth . 

[2]:effective but too-tepid biopic

In [3]:
class Generator():
    def __init__(self, input_list):
        self.input_list=input_list
    def __getitem__(self,item):
        return (np.array(self.input_list[item][0],dtype=np.int32),
                np.array(self.input_list[item][1],dtype=np.int32))
    def __len__(self):
        return len(self.input_list)


class MovieReview:
    '''
    影评数据集
    '''
    def __init__(self, root_dir, maxlen, split):
        '''
        input:
            root_dir: 影评数据目录
            maxlen: 设置句子最大长度
            split: 设置数据集中训练/评估的比例
        '''
        self.path = root_dir
        self.feelMap = {
            'neg':0,
            'pos':1
        }
        self.files = []

        self.doConvert = False
        
        mypath = Path(self.path)
        if not mypath.exists() or not mypath.is_dir():
            print("please check the root_dir!")
            raise ValueError

        # 在数据目录中找到文件
        for root,_,filename in os.walk(self.path):
            for each in filename:
                self.files.append(os.path.join(root,each))
            break

        # 确认是否为两个文件.neg与.pos
        if len(self.files) != 2:
            print("There are {} files in the root_dir".format(len(self.files)))
            raise ValueError



       # 读取数据
        self.word_num = 0
        self.maxlen = 0
        self.minlen = float("inf")
        self.maxlen = float("-inf")
        self.Pos = []
        self.Neg = []
        for filename in self.files:
            self.read_data(filename)

        self.text2vec(maxlen=maxlen)
        self.split_dataset(split=split)

    def read_data(self, filePath):
        with open(filePath, 'r', encoding='utf-8' ) as f:
            for sentence in f.readlines():
                sentence = sentence.replace('\n','')\
                                    .replace('"','')\
                                    .replace('\'','')\
                                    .replace('.','')\
                                    .replace(',','')\
                                    .replace('[','')\
                                    .replace(']','')\
                                    .replace('(','')\
                                    .replace(')','')\
                                    .replace(':','')\
                                    .replace('--','')\
                                    .replace('-',' ')\
                                    .replace('\\','')\
                                    .replace('0','')\
                                    .replace('1','')\
                                    .replace('2','')\
                                    .replace('3','')\
                                    .replace('4','')\
                                    .replace('5','')\
                                    .replace('6','')\
                                    .replace('7','')\
                                    .replace('8','')\
                                    .replace('9','')\
                                    .replace('`','')\
                                    .replace('=','')\
                                    .replace('$','')\
                                    .replace('/','')\
                                    .replace('*','')\
                                    .replace(';','')\
                                    .replace('<b>','')\
                                    .replace('%','')


                sentence = sentence.split(' ')
                sentence = list(filter(lambda x: x, sentence))
                if sentence:
                    self.word_num += len(sentence)
                    self.maxlen = self.maxlen if self.maxlen >= len(sentence) else len(sentence)
                    self.minlen = self.minlen if self.minlen <= len(sentence) else len(sentence)
                    if 'pos' in filePath:
                        self.Pos.append([sentence,self.feelMap['pos']])
                    else:
                        self.Neg.append([sentence,self.feelMap['neg']])

    def text2vec(self, maxlen):
        '''
# 将句子转化为向量
        '''
        # Vocab = {word : index}
        self.Vocab = dict()

        # self.Vocab['None']
        for SentenceLabel in self.Pos+self.Neg:
            vector = [0]*maxlen
            for index, word in enumerate(SentenceLabel[0]):
                if index >= maxlen:
                    break
                if word not in self.Vocab.keys():
                    self.Vocab[word] = len(self.Vocab)
                    vector[index] = len(self.Vocab) - 1
                else:
                    vector[index] = self.Vocab[word]
            SentenceLabel[0] = vector
        self.doConvert = True

    def split_dataset(self, split):
        '''
        分割为训练集与测试集

        '''
        trunk_pos_size = math.ceil((1-split)*len(self.Pos))
        trunk_neg_size = math.ceil((1-split)*len(self.Neg))
        trunk_num = int(1/(1-split))
        pos_temp=list()
        neg_temp=list()
        for index in range(trunk_num):
            pos_temp.append(self.Pos[index*trunk_pos_size:(index+1)*trunk_pos_size])
            neg_temp.append(self.Neg[index*trunk_neg_size:(index+1)*trunk_neg_size])
        self.test = pos_temp.pop(2)+neg_temp.pop(2)
        self.train = [i for item in pos_temp+neg_temp for i in item]

        random.shuffle(self.train)

    def get_dict_len(self):
        '''
        获得数据集中文字组成的词典长度
        '''
        if self.doConvert:
            return len(self.Vocab)
        else:
            print("Haven't finished Text2Vec")
            return -1

    def create_train_dataset(self, epoch_size, batch_size):
        dataset = ds.GeneratorDataset(
                                        source=Generator(input_list=self.train), 
                                        column_names=["data","label"], 
                                        shuffle=False
                                        )
        dataset=dataset.batch(batch_size=batch_size,drop_remainder=True)
        dataset=dataset.repeat(epoch_size)
        return dataset

    def create_test_dataset(self, batch_size):
        dataset = ds.GeneratorDataset(
                                        source=Generator(input_list=self.test), 
                                        column_names=["data","label"], 
                                        shuffle=False
                                        )
        dataset=dataset.batch(batch_size=batch_size,drop_remainder=True)
        return dataset

In [4]:
instance = MovieReview(root_dir=cfg.data_path, maxlen=cfg.word_len, split=0.9)
dataset = instance.create_train_dataset(batch_size=cfg.batch_size,epoch_size=cfg.epoch_size)
batch_num = dataset.get_dataset_size()

In [5]:
vocab_size=instance.get_dict_len()
print("vocab_size:{0}".format(vocab_size))
item =dataset.create_dict_iterator()
for i,data in enumerate(item):
    if i<1:
        print(data)
        print(data['data'][1])
    else:
        break

vocab_size:18848
{'data': Tensor(shape=[64, 51], dtype=Int32, value=
[[  930,   747,  9005 ...     0,     0,     0],
 [14452,   111,     0 ...     0,     0,     0],
 [ 1249,  1250,    10 ...     0,     0,     0],
 ...
 [   15,  4410, 10020 ...     0,     0,     0],
 [16064,    10,  8916 ...     0,     0,     0],
 [  443,    15,  2424 ...     0,     0,     0]]), 'label': Tensor(shape=[64], dtype=Int32, value= [0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 
 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 
 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0])}
[14452   111     0 17967    32    75  1628  2287   415     0  7626  1197
    32     0 17968    10     0  4092 17969    82  1938   920    55    82
   596  1644    15 10863  8950     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0]


# 配置训练参数

In [6]:
learning_rate = []
warm_up = [1e-3 / math.floor(cfg.epoch_size / 5) * (i + 1) for _ in range(batch_num) 
           for i in range(math.floor(cfg.epoch_size / 5))]
shrink = [1e-3 / (16 * (i + 1)) for _ in range(batch_num) 
          for i in range(math.floor(cfg.epoch_size * 3 / 5))]
normal_run = [1e-3 for _ in range(batch_num) for i in 
              range(cfg.epoch_size - math.floor(cfg.epoch_size / 5) 
                    - math.floor(cfg.epoch_size * 2 / 5))]
learning_rate = learning_rate + warm_up + normal_run + shrink

# TextCNN模型定义

In [7]:
def _weight_variable(shape, factor=0.01):
    init_value = np.random.randn(*shape).astype(np.float32) * factor
    return Tensor(init_value)

def make_conv_layer(kernel_size):
    weight_shape = (96, 1, *kernel_size)
    weight = _weight_variable(weight_shape)
    return nn.Conv2d(in_channels=1, out_channels=96, kernel_size=kernel_size, padding=1,
                     pad_mode="pad", weight_init=weight, has_bias=True)

class TextCNN(nn.Cell):
    def __init__(self, vocab_len, word_len, num_classes, vec_length):
        super(TextCNN, self).__init__()
        self.vec_length = vec_length
        self.word_len = word_len
        self.num_classes = num_classes

        self.unsqueeze = ops.ExpandDims()
        self.embedding = nn.Embedding(vocab_len, self.vec_length, embedding_table='normal')

        self.slice = ops.Slice()
        self.layer1 = self.make_layer(kernel_height=3)
        self.layer2 = self.make_layer(kernel_height=4)
        self.layer3 = self.make_layer(kernel_height=5)

        self.concat = ops.Concat(1)

        self.fc = nn.Dense(96*3, self.num_classes)
        self.drop = nn.Dropout(keep_prob=0.5)
        self.print = ops.Print()
        self.reducemean = ops.ReduceMax(keep_dims=False)
        
    def make_layer(self, kernel_height):
        return nn.SequentialCell(
            [
                make_conv_layer((kernel_height,self.vec_length)),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=(self.word_len-kernel_height+1,1)),
            ]
        )

    def construct(self,x):
        x = self.unsqueeze(x, 1)
        x = self.embedding(x)
        x1 = self.layer1(x)
        x2 = self.layer2(x)
        x3 = self.layer3(x)

        x1 = self.reducemean(x1, (2, 3))
        x2 = self.reducemean(x2, (2, 3))
        x3 = self.reducemean(x3, (2, 3))

        x = self.concat((x1, x2, x3))
        x = self.drop(x)
        x = self.fc(x)
        return x
net = TextCNN(vocab_len=instance.get_dict_len(), word_len=cfg.word_len, 
              num_classes=cfg.num_classes, vec_length=cfg.vec_length)
print(net)

[WARNING] ME(35820:23716,MainProcess):2023-12-28-02:33:16.871.550 [mindspore\nn\layer\basic.py:173] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.
[WARNING] ME(35820:23716,MainProcess):2023-12-28-02:33:16.875.684 [mindspore\nn\layer\basic.py:199] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.


TextCNN<
  (embedding): Embedding<vocab_size=18848, embedding_size=40, use_one_hot=False, embedding_table=Parameter (name=embedding.embedding_table, shape=(18848, 40), dtype=Float32, requires_grad=True), dtype=Float32, padding_idx=None>
  (layer1): SequentialCell<
    (0): Conv2d<input_channels=1, output_channels=96, kernel_size=(3, 40), stride=(1, 1), pad_mode=pad, padding=1, dilation=(1, 1), group=1, has_bias=True, weight_init=[[[[-1.6271515e-02  1.6023751e-03 -4.3710796e-03 ... -2.9781717e-03
         6.4188465e-03 -1.2429118e-03]
       [-7.0424764e-03 -1.1754459e-02  5.6295167e-03 ... -1.4293620e-04
         1.2338966e-02  1.4851415e-02]
       [ 1.7912695e-04 -3.8437906e-03  3.3114781e-03 ... -5.0975513e-03
        -4.4399868e-03  1.6553575e-02]]]
    
    
     [[[ 3.7730851e-03  1.5607016e-02 -8.3354581e-03 ...  1.5496569e-02
        -5.2156094e-03 -3.9543784e-03]
       [-2.4618681e-03 -2.2576971e-03  1.4365268e-03 ... -1.9940890e-02
        -2.2237515e-03 -1.2470160e-02]
    

# 定义训练的相关参数

In [8]:
# 优化器、损失函数、保存检查点、时间监视器等设置
opt = nn.Adam(filter(lambda x: x.requires_grad, net.get_parameters()), 
              learning_rate=learning_rate, weight_decay=cfg.weight_decay)
loss = nn.SoftmaxCrossEntropyWithLogits(sparse=True)
model = Model(net, loss_fn=loss, optimizer=opt, metrics={'acc': Accuracy()})
config_ck = CheckpointConfig(save_checkpoint_steps=int(cfg.epoch_size*batch_num/2), keep_checkpoint_max=cfg.keep_checkpoint_max)
time_cb = TimeMonitor(data_size=batch_num)
ckpt_save_dir = "./ckpt"
ckpoint_cb = ModelCheckpoint(prefix="train_textcnn", directory=ckpt_save_dir, config=config_ck)
loss_cb = LossMonitor()

# 启动训练

In [9]:
model.train(cfg.epoch_size, dataset, callbacks=[time_cb, ckpoint_cb, loss_cb])
print("train success")

[WARNING] ME(35820:23716,MainProcess):2023-12-28-02:33:16.930.140 [mindspore\nn\layer\basic.py:199] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.
[WARNING] ME(35820:23716,MainProcess):2023-12-28-02:33:16.971.624 [mindspore\nn\layer\basic.py:199] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.
[WARNING] ME(35820:23716,MainProcess):2023-12-28-02:33:16.980.255 [mindspore\nn\layer\basic.py:199] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.
[WARNING] ME(35820:23716,MainProcess):2023-12-28-02:33:17.111.73 [mindspore\nn\layer\basic.py:199] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.
[WARNING] ME(35820:23716,MainProcess):2023-12-28-02:33:17.151.73 [mindspore\nn\layer\basic.py:199] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.
[WARNING] ME(35820:23716,MainProcess):2023-12-28-02:33:17.261.07 [mindspore\nn\lay

epoch: 1 step: 1, loss is 0.6905139684677124
epoch: 1 step: 2, loss is 0.6879487633705139
epoch: 1 step: 3, loss is 0.7110053896903992
epoch: 1 step: 4, loss is 0.692635178565979
epoch: 1 step: 5, loss is 0.6989395618438721
epoch: 1 step: 6, loss is 0.6954412460327148
epoch: 1 step: 7, loss is 0.6904729604721069
epoch: 1 step: 8, loss is 0.7022413015365601
epoch: 1 step: 9, loss is 0.6911741495132446
epoch: 1 step: 10, loss is 0.6982966065406799
epoch: 1 step: 11, loss is 0.6935350298881531
epoch: 1 step: 12, loss is 0.6961201429367065
epoch: 1 step: 13, loss is 0.7010985612869263
epoch: 1 step: 14, loss is 0.6957391500473022
epoch: 1 step: 15, loss is 0.6840680241584778
epoch: 1 step: 16, loss is 0.6927784085273743
epoch: 1 step: 17, loss is 0.6916652917861938
epoch: 1 step: 18, loss is 0.6973812580108643
epoch: 1 step: 19, loss is 0.6943005323410034
epoch: 1 step: 20, loss is 0.6960018277168274
epoch: 1 step: 21, loss is 0.6899570226669312
epoch: 1 step: 22, loss is 0.693226158618927

epoch: 1 step: 177, loss is 0.47559940814971924
epoch: 1 step: 178, loss is 0.42776161432266235
epoch: 1 step: 179, loss is 0.4777320623397827
epoch: 1 step: 180, loss is 0.5014688372612
epoch: 1 step: 181, loss is 0.4548027515411377
epoch: 1 step: 182, loss is 0.6713588833808899
epoch: 1 step: 183, loss is 0.4653418958187103
epoch: 1 step: 184, loss is 0.4099550247192383
epoch: 1 step: 185, loss is 0.6182129979133606
epoch: 1 step: 186, loss is 0.37724238634109497
epoch: 1 step: 187, loss is 0.5401849746704102
epoch: 1 step: 188, loss is 0.46215134859085083
epoch: 1 step: 189, loss is 0.5023139715194702
epoch: 1 step: 190, loss is 0.3797301650047302
epoch: 1 step: 191, loss is 0.466511994600296
epoch: 1 step: 192, loss is 0.5010091066360474
epoch: 1 step: 193, loss is 0.5074562430381775
epoch: 1 step: 194, loss is 0.46580174565315247
epoch: 1 step: 195, loss is 0.404690146446228
epoch: 1 step: 196, loss is 0.4861869215965271
epoch: 1 step: 197, loss is 0.4383508265018463
epoch: 1 step

epoch: 1 step: 349, loss is 0.21508824825286865
epoch: 1 step: 350, loss is 0.24238212406635284
epoch: 1 step: 351, loss is 0.3434457778930664
epoch: 1 step: 352, loss is 0.23832833766937256
epoch: 1 step: 353, loss is 0.3794994652271271
epoch: 1 step: 354, loss is 0.29682835936546326
epoch: 1 step: 355, loss is 0.23333114385604858
epoch: 1 step: 356, loss is 0.19182419776916504
epoch: 1 step: 357, loss is 0.18579354882240295
epoch: 1 step: 358, loss is 0.2629631459712982
epoch: 1 step: 359, loss is 0.202119380235672
epoch: 1 step: 360, loss is 0.23200862109661102
epoch: 1 step: 361, loss is 0.19668744504451752
epoch: 1 step: 362, loss is 0.20372352004051208
epoch: 1 step: 363, loss is 0.15194883942604065
epoch: 1 step: 364, loss is 0.17228032648563385
epoch: 1 step: 365, loss is 0.24661508202552795
epoch: 1 step: 366, loss is 0.2640238106250763
epoch: 1 step: 367, loss is 0.17360778152942657
epoch: 1 step: 368, loss is 0.30952030420303345
epoch: 1 step: 369, loss is 0.1717076748609542

epoch: 1 step: 520, loss is 0.10334978997707367
epoch: 1 step: 521, loss is 0.17683430016040802
epoch: 1 step: 522, loss is 0.07161248475313187
epoch: 1 step: 523, loss is 0.060948166996240616
epoch: 1 step: 524, loss is 0.06406490504741669
epoch: 1 step: 525, loss is 0.06838170439004898
epoch: 1 step: 526, loss is 0.12353499233722687
epoch: 1 step: 527, loss is 0.059011951088905334
epoch: 1 step: 528, loss is 0.08835780620574951
epoch: 1 step: 529, loss is 0.0588945634663105
epoch: 1 step: 530, loss is 0.1218414455652237
epoch: 1 step: 531, loss is 0.08321259170770645
epoch: 1 step: 532, loss is 0.06818564236164093
epoch: 1 step: 533, loss is 0.13037075102329254
epoch: 1 step: 534, loss is 0.04139417037367821
epoch: 1 step: 535, loss is 0.0518244206905365
epoch: 1 step: 536, loss is 0.04368699714541435
epoch: 1 step: 537, loss is 0.13946695625782013
epoch: 1 step: 538, loss is 0.10792014002799988
epoch: 1 step: 539, loss is 0.13110856711864471
epoch: 1 step: 540, loss is 0.05801311135

...

epoch: 3 step: 498, loss is 0.0016292589716613293
epoch: 3 step: 499, loss is 0.0009034331305883825
epoch: 3 step: 500, loss is 0.0011520847911015153
epoch: 3 step: 501, loss is 0.0004716186667792499
epoch: 3 step: 502, loss is 0.003372551640495658
epoch: 3 step: 503, loss is 0.0020301807671785355
epoch: 3 step: 504, loss is 0.003246284555643797
epoch: 3 step: 505, loss is 0.0006146065425127745
epoch: 3 step: 506, loss is 0.0008742191712372005
epoch: 3 step: 507, loss is 0.001790322596207261
epoch: 3 step: 508, loss is 0.003313402645289898
epoch: 3 step: 509, loss is 0.0005966292810626328
epoch: 3 step: 510, loss is 0.0007230765768326819
epoch: 3 step: 511, loss is 0.0016204476123675704
epoch: 3 step: 512, loss is 0.0004956031916663051
epoch: 3 step: 513, loss is 0.0014012550236657262
epoch: 3 step: 514, loss is 0.002669316018000245
epoch: 3 step: 515, loss is 0.0005760216736234725
epoch: 3 step: 516, loss is 0.00032630161149427295
epoch: 3 step: 517, loss is 0.0011424827389419079
epoc

epoch: 4 step: 66, loss is 0.0005567899788729846
epoch: 4 step: 67, loss is 0.0013948141131550074
epoch: 4 step: 68, loss is 0.0006297611398622394
epoch: 4 step: 69, loss is 0.0003122533089481294
epoch: 4 step: 70, loss is 0.0004941242514178157
epoch: 4 step: 71, loss is 0.00045288639375939965
epoch: 4 step: 72, loss is 0.0007087915437296033
epoch: 4 step: 73, loss is 0.0004719663120340556
epoch: 4 step: 74, loss is 0.0015679383650422096
epoch: 4 step: 75, loss is 0.000869676296133548
epoch: 4 step: 76, loss is 0.0014896038919687271
epoch: 4 step: 77, loss is 0.0015248676063492894
epoch: 4 step: 78, loss is 0.00040246068965643644
epoch: 4 step: 79, loss is 0.002844941569492221
epoch: 4 step: 80, loss is 0.0008478400995954871
epoch: 4 step: 81, loss is 0.0007617830415256321
epoch: 4 step: 82, loss is 0.0002601441810838878
epoch: 4 step: 83, loss is 0.0005089112091809511
epoch: 4 step: 84, loss is 0.00038273382233455777
epoch: 4 step: 85, loss is 0.001331555307842791
epoch: 4 step: 86, l

epoch: 4 step: 230, loss is 0.0009553593117743731
epoch: 4 step: 231, loss is 0.0009844263549894094
epoch: 4 step: 232, loss is 0.0007499025668948889
epoch: 4 step: 233, loss is 0.000489596975967288
epoch: 4 step: 234, loss is 0.0009965298231691122
epoch: 4 step: 235, loss is 0.0016081726644188166
epoch: 4 step: 236, loss is 0.0006266592536121607
epoch: 4 step: 237, loss is 0.0003915818524546921
epoch: 4 step: 238, loss is 0.001185113564133644
epoch: 4 step: 239, loss is 0.00054706702940166
epoch: 4 step: 240, loss is 0.001825604122132063
epoch: 4 step: 241, loss is 0.0009344199206680059
epoch: 4 step: 242, loss is 0.0012470833025872707
epoch: 4 step: 243, loss is 0.0008201023447327316
epoch: 4 step: 244, loss is 0.0008112993673421443
epoch: 4 step: 245, loss is 0.0008138916455209255
epoch: 4 step: 246, loss is 0.000543452799320221
epoch: 4 step: 247, loss is 0.00018352306506130844
epoch: 4 step: 248, loss is 0.0006264058174565434
epoch: 4 step: 249, loss is 0.0005232341354712844
epoch

epoch: 4 step: 395, loss is 0.0002758324844762683
epoch: 4 step: 396, loss is 0.0001880207855720073
epoch: 4 step: 397, loss is 0.0002822804090101272
epoch: 4 step: 398, loss is 0.0005809561116620898
epoch: 4 step: 399, loss is 0.0003739330277312547
epoch: 4 step: 400, loss is 0.00021893659140914679
epoch: 4 step: 401, loss is 0.000584094668738544
epoch: 4 step: 402, loss is 0.001151426462456584
epoch: 4 step: 403, loss is 0.0016428872477263212
epoch: 4 step: 404, loss is 0.000541126704774797
epoch: 4 step: 405, loss is 0.0007187669398263097
epoch: 4 step: 406, loss is 0.00042706335079856217
epoch: 4 step: 407, loss is 0.000886803783942014
epoch: 4 step: 408, loss is 0.0016704356530681252
epoch: 4 step: 409, loss is 0.0015116669237613678
epoch: 4 step: 410, loss is 0.0003146606031805277
epoch: 4 step: 411, loss is 0.0007897259201854467
epoch: 4 step: 412, loss is 0.001736731268465519
epoch: 4 step: 413, loss is 0.000626474036835134
epoch: 4 step: 414, loss is 0.0006236432818695903
epoc

epoch: 4 step: 559, loss is 0.00038954601041041315
epoch: 4 step: 560, loss is 0.0010648492025211453
epoch: 4 step: 561, loss is 0.003239247016608715
epoch: 4 step: 562, loss is 0.00047454365994781256
epoch: 4 step: 563, loss is 0.0008217727299779654
epoch: 4 step: 564, loss is 0.0028817495331168175
epoch: 4 step: 565, loss is 0.0003736595681402832
epoch: 4 step: 566, loss is 0.00031737738754600286
epoch: 4 step: 567, loss is 0.001035381923429668
epoch: 4 step: 568, loss is 0.0012446609325706959
epoch: 4 step: 569, loss is 0.0005712719284929335
epoch: 4 step: 570, loss is 0.0012011248618364334
epoch: 4 step: 571, loss is 0.0006780567928217351
epoch: 4 step: 572, loss is 0.0006606008391827345
epoch: 4 step: 573, loss is 0.0016953303711488843
epoch: 4 step: 574, loss is 0.0007782676257193089
epoch: 4 step: 575, loss is 0.0021071825176477432
epoch: 4 step: 576, loss is 0.0013762351591140032
epoch: 4 step: 577, loss is 0.0010777077404782176
epoch: 4 step: 578, loss is 0.0003406903124414384

# 测试评估

In [10]:
def preprocess(sentence):
    sentence = sentence.lower().strip()
    sentence = sentence.replace('\n','')\
                                    .replace('"','')\
                                    .replace('\'','')\
                                    .replace('.','')\
                                    .replace(',','')\
                                    .replace('[','')\
                                    .replace(']','')\
                                    .replace('(','')\
                                    .replace(')','')\
                                    .replace(':','')\
                                    .replace('--','')\
                                    .replace('-',' ')\
                                    .replace('\\','')\
                                    .replace('0','')\
                                    .replace('1','')\
                                    .replace('2','')\
                                    .replace('3','')\
                                    .replace('4','')\
                                    .replace('5','')\
                                    .replace('6','')\
                                    .replace('7','')\
                                    .replace('8','')\
                                    .replace('9','')\
                                    .replace('`','')\
                                    .replace('=','')\
                                    .replace('$','')\
                                    .replace('/','')\
                                    .replace('*','')\
								 .replace(';','')\
                                    .replace('<b>','')\
                                    .replace('%','')\
                                    .replace("  "," ")
    sentence = sentence.split(' ')
    maxlen = cfg.word_len
    vector = [0]*maxlen
    for index, word in enumerate(sentence):
        if index >= maxlen:
            break
        if word not in instance.Vocab.keys():
            print(word,"单词未出现在字典中")
        else:
            vector[index] = instance.Vocab[word]
    sentence = vector

    return sentence

def inference(review_en):
    review_en = preprocess(review_en)
    input_en = Tensor(np.array([review_en]).astype(np.int32))
    output = net(input_en)
    if np.argmax(np.array(output[0])) == 1:
        print("Positive comments")
    else:
        print("Negative comments")

In [26]:
review_en = "It is a very good experience which blows my mind and charges me to full energy"
inference(review_en)

Positive comments


# 思考题

如果想通过GPU跑以上实验，需要在哪个地方修改相关配置为GPU？
在开头cfg配置中的device_target处更改，将CPU改为GPU即可